In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("../data/AmesHousing.csv", sep='\t')

# Basic dimensions & first glance
print(df.shape)
df.head()

In [ ]:
# Updated list with exact CSV column names
cols_na_means_none = [
    'Alley', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure', 
    'BsmtFin Type 1', 'BsmtFin Type 2', 'Fireplace Qu', 
    'Garage Type', 'Garage Finish', 'Garage Qual', 
    'Garage Cond', 'Pool QC', 'Fence', 'Misc Feature'
]

# Replace NaN with 'None' for these specific columns
df[cols_na_means_none] = df[cols_na_means_none].fillna('None')

In [ ]:
# Identify numerical and categorical columns
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()

print(f"Numerical columns: {len(num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

# Explicitly display summary statistics
display(df.describe().round(2))

# Check missing values
missing = df.isnull().sum()
missing_sorted = missing[missing > 0].sort_values(ascending=False)
display(missing_sorted)

print(f"Dataset Timespan: {df['Yr Sold'].min()}, {df['Yr Sold'].max()}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Cum sunt distribuite vanzarile in Ames?

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['SalePrice'], kde=True, color='royalblue', bins=40)
plt.title('Distribution of SalePrice', fontsize=14)
plt.xlabel('Sale Price ($)')
plt.ylabel('Frequency')
plt.show()

print(f"Skewness: {df['SalePrice'].skew():.2f}")

## 2. Care atribute au cea mai mare corelatie liniara cu pretul?

In [ ]:
plt.figure(figsize=(10, 8))
# Select the top 10 numerical features most correlated with SalePrice
top_corr_features = df[num_cols].corr()['SalePrice'].abs().sort_values(ascending=False).head(10).index
corr_matrix = df[top_corr_features].corr()

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', cbar=True, square=True)
plt.title('Heatmap: Top Correlated Features with SalePrice', fontsize=14)
plt.show()

## 3. Pretul creste liniar cu aria proprietatii?

In [ ]:
plt.figure(figsize=(10, 6))
df_cleaned = df.drop(df[(df['Gr Liv Area'] > 4000) & ((df['SalePrice'] < 300000))].index)

# Zeichnet den Scatter-Plot inklusive linearer Regressionslinie
sns.regplot(
    data=df_cleaned, 
    x='Gr Liv Area', 
    y='SalePrice', 
    scatter_kws={'alpha': 0.5, 'color': 'teal'}, 
    line_kws={'color': 'darkorange', 'linewidth': 2, 'label': 'Trendlinie (OLS)'}
)

# Schwellenwert-Linie für die bekannten Ames-Ausreißer (> 4000 sq ft)
plt.axvline(4000, color='red', linestyle='--', label='Outlier-Grenze (> 4000 sq ft)')

plt.title('Above Grade Living Area (Gr Liv Area) vs. SalePrice mit Trendlinie', fontsize=14)
plt.xlabel('Ground Living Area (sq ft)')
plt.ylabel('Sale Price ($)')
plt.legend()
plt.show()

## 4. Cum afecteaza scorul de calitate mediana si variatia pretului?

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, hue='Overall Qual', y='SalePrice', palette='Blues')
plt.title('SalePrice Distribution across Overall Quality Ratings (1-10)', fontsize=14)
plt.xlabel('Overall Quality')
plt.ylabel('Sale Price ($)')
plt.show()

## 5. Care este impactul cartierului (Neighborhood) asupra prețului de vânzare?
*Interpretare: Acest Violin Plot arată densitatea și distribuția prețurilor în funcție de cartier. Putem observa că unele cartiere (ex. NoRidge, NridgHt) au o mediană mult mai mare și o variație extinsă a prețurilor, în timp ce altele (ex. OldTown) sunt mai dense la un nivel de preț mai scăzut.*

In [ ]:
plt.figure(figsize=(14, 7))
sns.violinplot(
    data=df, 
    x='Neighborhood', 
    y='SalePrice', 
    hue='Neighborhood',
    palette='muted',
    legend=False
)
plt.xticks(rotation=90)
plt.title('Sale Price Distribution by Neighborhood', fontsize=14)
plt.xlabel('Neighborhood')
plt.ylabel('Sale Price ($)')
plt.tight_layout()
plt.show()

## 6. Casele mai noi sunt mai scumpe? (Year Built vs SalePrice)
*Interpretare: Aici analizăm relația dintre anul construcției și prețul de vânzare. Observăm o tendință generală de creștere a prețului pentru casele construite recent (după 1980). De asemenea, casele foarte vechi (înainte de 1920) au o fluctuație interesantă, probabil datorită renovărilor istorice.*

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='Year Built', 
    y='SalePrice', 
    alpha=0.5, 
    color='purple'
)
plt.title('Impact of Construction Year on Sale Price', fontsize=14)
plt.xlabel('Year Built')
plt.ylabel('Sale Price ($)')
plt.show()

## 7. Aerul condiționat centralizat crește valoarea proprietății?
*Interpretare: Utilizând un Box Plot, comparăm medianele și valorile aberante (outliers) pentru casele cu și fără aer condiționat centralizat (Central Air). Proprietățile cu aer condiționat (Y) au clar o valoare mediană de vânzare superioară și ating prețuri mult mai mari.*

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df, 
    x='Central Air', 
    y='SalePrice', 
    hue='Central Air',
    palette='Set2'
)
plt.title('Sale Price Differences: Central Air vs. No Central Air', fontsize=14)
plt.xlabel('Central Air (Y = Yes, N = No)')
plt.ylabel('Sale Price ($)')
plt.show()

## 8. Cum influențează suprafața totală a subsolului prețul?
*Interpretare: Acest grafic ilustrează corelația dintre suprafața totală a subsolului (`Total Bsmt SF`) și prețul de vânzare. Se observă o corelație pozitivă puternică. Casele cu subsol de peste 3000 sq ft sunt extrem de rare și pot fi considerate outliers.*

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df, 
    x='Total Bsmt SF', 
    y='SalePrice', 
    alpha=0.4, 
    color='crimson'
)
plt.title('Total Basement Area vs. Sale Price', fontsize=14)
plt.xlabel('Total Basement Square Footage')
plt.ylabel('Sale Price ($)')
plt.show()